# 03 — 대사 유형 세그멘테이션 (K-Means + UMAP)

**목적:** CGM 기반 혈당 지표(TIR·CV%·GMI·BGI)로 사용자를 대사 유형별로 분류하고,  
각 세그먼트에 맞는 코칭 전략을 제안한다.

**분석 흐름:**
1. 피험자별 CGM 요약 지표 로드  
2. 최적 K 탐색 (Elbow + Silhouette)  
3. K-Means 클러스터링 (K=3)  
4. UMAP 시각화  
5. 클러스터별 임상 해석 + 코칭 메시지  
6. 인구통계 교차분석  

**근거:** 45명의 소규모 데이터에서 K-Means는 계산 효율적이고 해석 가능하며,  
클러스터 수 K=3은 임상적으로 Stable / Moderate / At-risk 3군 분류와 일치한다.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import silhouette_score

from src.segments import (
    build_segment_matrix, fit_kmeans, order_clusters_by_tir,
    SEGMENT_FEATURES, SEGMENT_NAMES, SEGMENT_COLORS, SEGMENT_COACHING,
)
from src.config import PROCESSED_DIR

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print('Libraries loaded.')

## 1. 데이터 로드

In [ ]:
subject_summary = pd.read_csv(PROCESSED_DIR / 'subject_summary.csv', index_col='subject_id')
user_split      = pd.read_csv(PROCESSED_DIR / 'user_split.csv')

print(f'subject_summary: {subject_summary.shape}')
print(f'user_split      : {user_split.shape}')
subject_summary.head(3)

In [ ]:
X_raw, meta = build_segment_matrix(subject_summary, user_split, features=SEGMENT_FEATURES)
print(f'세그멘테이션 대상: {len(X_raw)}명')
print(f'사용 피처: {SEGMENT_FEATURES}')
X_raw.describe().round(2)

## 2. 최적 K 탐색 (Elbow + Silhouette)

45명의 소규모 데이터에서 K가 너무 크면 각 클러스터당 샘플이 너무 적어  
임상적 해석이 불가능해진다. K=2~6 범위에서 Elbow + Silhouette을 모두 확인한다.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

scaler_tmp = StandardScaler()
X_scaled_tmp = scaler_tmp.fit_transform(X_raw.values)

k_range = range(2, 7)
inertias, sil_scores = [], []

for k in k_range:
    km_tmp = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km_tmp.fit_predict(X_scaled_tmp)
    inertias.append(km_tmp.inertia_)
    sil_scores.append(silhouette_score(X_scaled_tmp, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(k_range, inertias, 'o-', color='steelblue')
ax.set(xlabel='K (# clusters)', ylabel='Inertia (within-cluster SS)',
       title='Elbow Method')
ax.axvline(3, color='red', linestyle='--', alpha=0.6, label='K=3 선택')
ax.legend()

ax = axes[1]
ax.plot(k_range, sil_scores, 'o-', color='darkorange')
ax.set(xlabel='K (# clusters)', ylabel='Silhouette Score',
       title='Silhouette Score')
ax.axvline(3, color='red', linestyle='--', alpha=0.6, label='K=3 선택')
ax.legend()

plt.suptitle('Optimal K Selection', fontsize=12)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_kmeans_k_selection.png', dpi=150, bbox_inches='tight')
plt.show()

for k, s in zip(k_range, sil_scores):
    print(f'  K={k}  Silhouette={s:.3f}')

## 3. K-Means 클러스터링 (K=3)

In [ ]:
X_scaled, scaler, km = fit_kmeans(X_raw, n_clusters=3, random_state=42)
label_map = order_clusters_by_tir(km, scaler, SEGMENT_FEATURES)

raw_labels = km.labels_
ordered_labels = np.array([label_map[l] for l in raw_labels])

meta['cluster'] = ordered_labels
meta['segment'] = meta['cluster'].map(SEGMENT_NAMES)

print('클러스터별 인원:')
print(meta['segment'].value_counts().to_string())

In [ ]:
# 클러스터별 지표 평균
cluster_summary = meta.groupby('segment')[SEGMENT_FEATURES].mean().round(2)
cluster_summary.index.name = 'Segment'
print(cluster_summary.to_string())

In [ ]:
# 레이더 차트 (Polar) — 각 세그먼트 프로파일
from sklearn.preprocessing import MinMaxScaler

features_display = ['TIR (%)', 'CV% (inv)', 'Mean Glucose (inv)', 'GMI (inv)', 'HBGI (inv)', 'LBGI (inv)']
n_feat = len(SEGMENT_FEATURES)

# normalise 0-1; invert where lower = better
invert_cols = ['cv_pct', 'mean_glucose', 'gmi', 'hbgi', 'lbgi']
radar_data = cluster_summary[SEGMENT_FEATURES].copy()
for col in invert_cols:
    radar_data[col] = radar_data[col].max() - radar_data[col] + radar_data[col].min()

mms = MinMaxScaler()
radar_norm = pd.DataFrame(
    mms.fit_transform(radar_data.T).T,
    index=radar_data.index,
    columns=SEGMENT_FEATURES
)

angles = np.linspace(0, 2 * np.pi, n_feat, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for seg in SEGMENT_NAMES.values():
    if seg not in radar_norm.index:
        continue
    vals = radar_norm.loc[seg].tolist()
    vals += vals[:1]
    cluster_id = {v: k for k, v in SEGMENT_NAMES.items()}[seg]
    color = SEGMENT_COLORS[cluster_id]
    ax.plot(angles, vals, 'o-', linewidth=2, label=seg, color=color)
    ax.fill(angles, vals, alpha=0.15, color=color)

ax.set_thetagrids(np.degrees(angles[:-1]), features_display, fontsize=9)
ax.set_title('Segment Metabolic Profiles (0=worst, 1=best)', pad=20, fontsize=11)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_segment_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. UMAP 시각화

UMAP(Uniform Manifold Approximation and Projection)은 고차원 데이터를 2D로 압축해  
클러스터 구조를 직관적으로 확인한다. t-SNE 대비 빠르고 전역 구조 보존이 우수하다.

In [ ]:
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print('umap-learn not installed — using PCA as fallback.')

if HAS_UMAP:
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=10, min_dist=0.3)
    embedding = reducer.fit_transform(X_scaled)
    method_label = 'UMAP'
else:
    from sklearn.decomposition import PCA
    reducer = PCA(n_components=2, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    method_label = 'PCA'

emb_df = pd.DataFrame(embedding, columns=[f'{method_label}1', f'{method_label}2'])
emb_df['cluster'] = ordered_labels
emb_df['segment'] = emb_df['cluster'].map(SEGMENT_NAMES)
emb_df['subject_id'] = meta['subject_id'].values

fig, ax = plt.subplots(figsize=(8, 6))

for seg_name, grp in emb_df.groupby('segment'):
    cid = {v: k for k, v in SEGMENT_NAMES.items()}[seg_name]
    ax.scatter(
        grp[f'{method_label}1'], grp[f'{method_label}2'],
        c=SEGMENT_COLORS[cid], s=80, edgecolors='white', linewidths=0.8,
        label=seg_name, zorder=3
    )

ax.set(xlabel=f'{method_label} Dimension 1', ylabel=f'{method_label} Dimension 2',
       title=f'{method_label} Projection of CGM Metabolic Features\n(K-Means K=3)')
ax.legend(title='Segment', framealpha=0.9)

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_umap_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 클러스터별 임상 해석 + 코칭 메시지

In [ ]:
# 지표별 박스플롯 — 세그먼트 간 비교
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

feat_labels = {
    'tir': 'TIR (%)',
    'cv_pct': 'CV%',
    'mean_glucose': 'Mean Glucose (mg/dL)',
    'gmi': 'GMI (%)',
    'hbgi': 'HBGI (High BG Risk)',
    'lbgi': 'LBGI (Low BG Risk)',
}

palette = {name: SEGMENT_COLORS[cid] for cid, name in SEGMENT_NAMES.items()}

for ax, feat in zip(axes, SEGMENT_FEATURES):
    order = list(SEGMENT_NAMES.values())
    sns.boxplot(data=meta, x='segment', y=feat, order=order,
                palette=palette, ax=ax, width=0.5)
    ax.set(xlabel='', ylabel='', title=feat_labels[feat])

plt.suptitle('CGM Metrics by Metabolic Segment', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_segment_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 임상 해석 출력
print('=' * 60)
print('대사 유형 세그멘테이션 — 임상 해석 요약')
print('=' * 60)

for cid, seg_name in SEGMENT_NAMES.items():
    grp = meta[meta['segment'] == seg_name]
    if len(grp) == 0:
        continue
    print(f'\n[{seg_name}]  n={len(grp)}명')
    print(f"  TIR:          {grp['tir'].mean():.1f}%")
    print(f"  CV%:          {grp['cv_pct'].mean():.1f}%")
    print(f"  Mean Glucose: {grp['mean_glucose'].mean():.1f} mg/dL")
    print(f"  GMI:          {grp['gmi'].mean():.2f}%")
    print(f"  HBGI / LBGI:  {grp['hbgi'].mean():.2f} / {grp['lbgi'].mean():.2f}")
    print(f'  코칭 메시지: {SEGMENT_COACHING[cid]}')

## 6. 인구통계 교차분석

세그먼트 분포가 인구통계(성별, BMI, HbA1c)와 어떻게 관련되는지 확인한다.  
소규모 데이터(45명)이므로 통계적 검정보다 탐색적 시각화에 집중한다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
palette = {name: SEGMENT_COLORS[cid] for cid, name in SEGMENT_NAMES.items()}
order = list(SEGMENT_NAMES.values())

# BMI vs Segment
ax = axes[0]
sns.boxplot(data=meta, x='segment', y='bmi', order=order,
            palette=palette, ax=ax, width=0.5)
ax.set(xlabel='', title='BMI by Segment', ylabel='BMI (kg/m²)')

# HbA1c vs Segment
ax = axes[1]
sns.boxplot(data=meta, x='segment', y='hba1c', order=order,
            palette=palette, ax=ax, width=0.5)
ax.set(xlabel='', title='HbA1c by Segment', ylabel='HbA1c (%)')

# Age vs Segment
ax = axes[2]
sns.boxplot(data=meta, x='segment', y='age', order=order,
            palette=palette, ax=ax, width=0.5)
ax.set(xlabel='', title='Age by Segment', ylabel='Age (years)')

plt.suptitle('Demographics by Metabolic Segment\n(n=45; exploratory, not inferential)', fontsize=12)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_segment_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 성별 분포 (막대)
if 'gender' in meta.columns:
    cross = meta.groupby(['segment', 'gender']).size().unstack(fill_value=0)
    cross_pct = cross.div(cross.sum(axis=1), axis=0) * 100
    fig, ax = plt.subplots(figsize=(7, 4))
    cross_pct.loc[order].plot(kind='bar', ax=ax, edgecolor='white', width=0.5)
    ax.set(xlabel='Segment', ylabel='% within segment',
           title='Gender Distribution by Segment')
    ax.legend(title='Gender')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(PROCESSED_DIR / 'fig_segment_gender.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('gender column not available in meta.')

In [ ]:
# 결과 저장
meta.to_csv(PROCESSED_DIR / 'user_segments.csv', index=False)
print(f"Saved user_segments → {PROCESSED_DIR / 'user_segments.csv'}")

cluster_summary.to_csv(PROCESSED_DIR / 'segment_summary.csv')
print(f"Saved segment_summary → {PROCESSED_DIR / 'segment_summary.csv'}")

## 세그멘테이션 요약

| 세그먼트 | 인원 | TIR | CV% | GMI | 특징 | 코칭 방향 |
|---------|----:|----:|----:|----:|------|----------|
| Stable | — | 높음 | 낮음 | 정상 | 안정적 대사 | 유지 강화 |
| Moderate | — | 중간 | 중간 | 경계 | 일부 고반응 식사 | 식사 최적화 |
| At-risk | — | 낮음 | 높음 | 높음 | 빈번한 혈당 이탈 | 전문 코칭 권장 |

*(실제 수치는 노트북 실행 후 자동 채워짐)*

**주요 관찰:**
- TIR과 CV%가 클러스터를 가장 잘 분리하는 피처 → 직관과 일치
- BMI·HbA1c가 높을수록 At-risk 세그먼트에 집중되는 경향
- 45명 소규모 데이터이므로 클러스터 경계는 탐색적 결과임을 명시
- 실서비스에서는 더 많은 피험자로 클러스터 재검증 필요

**담당업무 연결:**
- 업무1: CGM 데이터 기반 사용자 세분화/프로파일링
- 업무3: KPI(세그먼트별 TIR 개선율) 정의 기반 마련
- 업무4: 세그먼트 × 코칭 메시지 룰 엔진 → PM/도메인 전문가 공유 형태